# Química Compleja y Zonas Múltiples
Este tutorial muestra cómo inyectar funciones de Python para modelar química altamente compleja.


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../../src'))
from pa3py.data import load_tripodpy_hdf5
from pa3py.pebble_accretion import PebbleAccretionModule3
from pa3py.composition import FunctionComposition
from pa3py import constants as c


## 1. Definiendo tu propia física química
PA3Py te permite definir una función simple `f(r, t)` que retorne diccionarios de abundancias.


In [ ]:
def quimica_multizona(r_cm, t_sec):
    # 1. Calculamos las snowlines en el tiempo actual
    # H2O decae con el tiempo, CO2 y CO son estáticas para este ejemplo
    r_h2o = 2.73 * c.AU * (max(t_sec, 1e-6) / 1e13)**(-0.5)
    r_co2 = 5.0 * c.AU
    r_co  = 12.0 * c.AU
    
    # 2. Asignamos abundancias dependiendo de dónde está el embrión (r_cm)
    if r_cm < r_h2o:
        return {'silicatos': 1.0}  # Zona rocosa pura
    elif r_cm < r_co2:
        return {'silicatos': 0.5, 'H2O': 0.5}  # Zona agua
    elif r_cm < r_co:
        return {'silicatos': 0.3, 'H2O': 0.3, 'CO2': 0.4} # Zona CO2
    else:
        return {'silicatos': 0.2, 'H2O': 0.2, 'CO2': 0.3, 'CO': 0.3} # Zona CO


## 2. Inyectar el modelo en la simulación


In [ ]:
data_path = '../../tests/test_data/run_smooth_a0.001_v10'
disk = load_tripodpy_hdf5(data_path)

# Se debe declarar explícitamente qué especies químicas podría retornar la función
mi_modelo = FunctionComposition(
    user_func=quimica_multizona, 
    species=['silicatos', 'H2O', 'CO2', 'CO']
)

sim = PebbleAccretionModule3(disk, comp_model=mi_modelo)

# Corremos en varias regiones para ver los resultados distintos
resultados = sim.run_growth([2.0, 4.0, 8.0, 15.0], M0_g=1e-3 * sim.M_EARTH)
sim.summary(resultados)
